# VAD Values Analysis for MSP-Podcast

This notebook computes the mean Valence, Activation, and Dominance for each emotion category in the MSP-Podcast utterances and displays them in PAD space. The categories "Other" and "Unspecified" are ignored.

In [ ]:
import pandas as pd

# Read the CSV file
df = pd.read_csv('C:/Datasets/MSP-PODCAST-Publish-2.0/Labels/labels_consensus.csv')

# Exclude 'X' and 'O' from EmoClass
df_filtered = df[~df['EmoClass'].isin(['X', 'O'])]

# Group by EmoClass and calculate the mean for EmoAct, EmoVal, and EmoDom
emotion_stats = df_filtered.groupby('EmoClass')[['EmoAct', 'EmoVal', 'EmoDom']].agg(['mean', 'count'])

# Flatten the column names for better readability
emotion_stats.columns = ['_'.join(col).strip() for col in emotion_stats.columns]

# Summary
summary_df = df_filtered.groupby('EmoClass')[['EmoAct', 'EmoVal', 'EmoDom']].mean().round(3)
summary_df['Count'] = df_filtered.groupby('EmoClass').size()

# Create emotion name mapping
emotion_names = {
    'A': 'Angry',
    'S': 'Sad',
    'H': 'Happy',
    'U': 'Surprise',
    'F': 'Fear',
    'D': 'Disgust',
    'C': 'Contempt',
    'N': 'Neutral'
}

# Create new index with emotion names in parentheses
new_index = [emotion_names[emo] for emo in summary_df.index]
summary_df.index = new_index
summary_df.to_csv('logs/msp_emotion_mean-values.csv')
print("\n\nMean Values for each emotion:")
print('-'*50)
print(summary_df)

## 2D Scatters

The following cell outputs a 2D scatter for each dimension pair, for each of the discrete categories.

TODO: Make a class for aesthetic attributes for each cattegory.

In [ ]:
import matplotlib.pyplot as plt
from mpl_toolkits import mplot3d
import pandas as pd

# Read the CSV file
df = pd.read_csv('C:/Datasets/MSP-PODCAST-Publish-2.0/Labels/labels_consensus.csv')
# Exclude 'X' and 'O' from EmoClass
df_filtered = df[~df['EmoClass'].isin(['X', 'O'])]

color_map = {
    'S': '#00ffb8', 
    'C': '#7c33ef', 
    'D': '#ff16c9', 
    'H': '#ff5656', 
    'N': '#2d2d2d', 
    'A': '#ffa900',
    'F': '#0b40ae', 
    'U': '#64ff16'
}

emotion_names = {
    'A': 'Angry',
    'S': 'Sad',
    'H': 'Happy',
    'U': 'Surprise',
    'F': 'Fear',
    'D': 'Disgust',
    'C': 'Contempt',
    'N': 'Neutral'
}

for emotion in color_map.items():
    emo_class = emotion[0]
    emo_color = emotion[1]
    emo_name = emotion_names[emo_class]
    
    # Filter for the current emotion
    emo_utterances = df_filtered[df_filtered['EmoClass'] == emo_class]

    # Create a figure and a set of subplots
    cm = 1/2.54
    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(30*cm, 10*cm))

    # First subplot: Valence vs. Activation
    ax1.scatter(emo_utterances['EmoVal'], emo_utterances['EmoAct'], c=emo_color, alpha=0.1)
    ax1.set_title('Valence vs. Activation for %s' % emo_name)
    ax1.set_xlabel('Valence')
    ax1.set_ylabel('Activation')
    ax1.grid(True)
    ax1.set_xlim(1, 7)
    ax1.set_ylim(1, 7)

    # Second subplot: Valence vs. Dominance
    ax2.scatter(emo_utterances['EmoVal'], emo_utterances['EmoDom'], c=emo_color,  alpha=0.1)
    ax2.set_title('Valence vs. Dominance for %s' % emo_name)
    ax2.set_xlabel('Valence')
    ax2.set_ylabel('Dominance')
    ax2.grid(True)
    ax2.set_xlim(1, 7)
    ax2.set_ylim(1, 7)

    # Third subplot: Activation vs. Dominance
    ax3.scatter(emo_utterances['EmoAct'], emo_utterances['EmoDom'], c=emo_color,  alpha=0.1)
    ax3.set_title('Activation vs. Dominance for %s' % emo_name)
    ax3.set_xlabel('Activation')
    ax3.set_ylabel('Dominance')
    ax3.grid(True)
    ax3.set_xlim(1, 7)
    ax3.set_ylim(1, 7)

    # Adjust layout and show the plot
    plt.tight_layout()
    plt.show()
    plt.savefig('logs/msp_%s_2Dscatter.png' % emo_name, dpi=300)
      


## Interactive Plot

Requires the last cell to be run once to generate the csv with the values for each category. Displays the mean value of each categorical label (excluding O and X).

In [ ]:
%matplotlib widget

import matplotlib.pyplot as plt
from mpl_toolkits import mplot3d
import pandas as pd

summary_df = pd.read_csv('logs/msp_emotion_summary.csv', index_col=0)

# Create 3D plot
cm = 1/2.54
fig = plt.figure(figsize=(18*cm, 18*cm))
ax = fig.add_subplot(111, projection='3d')

# Extract coordinates (excluding Count column)
x = summary_df['EmoAct']
y = summary_df['EmoVal'] 
z = summary_df['EmoDom']

# Define color map for emotions
color_map = {
    'Sad': '#00ffb8', 
    'Contempt': '#7c33ef', 
    'Disgust': '#ff16c9', 
    'Happy': '#ff5656', 
    'Neutral': '#2d2d2d', 
    'Angry': '#ffa900',
    'Fear': '#0b40ae', 
    'Surprise': '#64ff16'
}

# Map emotion names to colors
colors = [color_map[emotion] for emotion in summary_df.index]

# Create scatter plot
scatter = ax.scatter(x, y, z, s=100, c=colors, alpha=1)

# Add labels for each point
for i, emotion in enumerate(summary_df.index):
    ax.text(x.iloc[i], y.iloc[i], z.iloc[i], f'  {emotion}', fontsize=9)

# Create legend
#from matplotlib.lines import Line2D
#legend_elements = [Line2D([0], [0], marker='o', color='w', label=category, markerfacecolor=color, markersize=10)
#                   for category, color in color_map.items()]
#
#ax.legend(handles=legend_elements, bbox_to_anchor=(1.1, 1), loc='upper right')

# Set axis labels and limits
ax.set_xlabel('Activation')
ax.set_ylabel('Valence')
ax.set_zlabel('Dominance')
ax.set_xlim(1, 7)
ax.set_ylim(1, 7)
ax.set_zlim(1, 7)

# Set title
ax.set_title('MSP-Podcast', fontsize=12)

plt.tight_layout()
plt.show()